# 06 — Data Export for Dashboard

Prepares final aggregated tables for Tableau. Pre-computes all metrics so Tableau visualizes, not computes.

Requires `cms_data.db` from `01_setup.ipynb` and analysis from `03-05`.

In [ ]:
import sqlite3
import pandas as pd
from pathlib import Path

In [ ]:
DB = Path("../cms_data.db")
EXPORT = Path("../data/exports")
conn = sqlite3.connect(DB)

---
## 1. DRG-level summary (cost + readmission + utilization)

In [ ]:
drg_summary = pd.read_sql("""
WITH drg_costs AS (
  SELECT
    CLM_DRG_CD,
    COUNT(*) as num_claims,
    ROUND(SUM(CLM_PMT_AMT), 0) as total_cost,
    ROUND(AVG(CLM_PMT_AMT), 0) as avg_cost_per_claim,
    ROUND(AVG(CLM_UTLZTN_DAY_CNT), 1) as avg_los
  FROM inpatient_claims
  WHERE CLM_PMT_AMT > 0
  GROUP BY CLM_DRG_CD
),
drg_readmit AS (
  SELECT
    CLM_DRG_CD,
    SUM(CASE WHEN days_to_readmission > 0 AND days_to_readmission <= 30 THEN 1 ELSE 0 END) as readmissions_30d,
    COUNT(*) as total_admits
  FROM (
    SELECT
      CLM_DRG_CD,
      CAST((julianday(next_admsn) - julianday(dschg_dt)) AS INTEGER) as days_to_readmission
    FROM (
      SELECT
        CLM_DRG_CD,
        NCH_BENE_DSCHRG_DT as dschg_dt,
        LEAD(CLM_ADMSN_DT) OVER (PARTITION BY DESYNPUF_ID ORDER BY CLM_ADMSN_DT) as next_admsn
      FROM inpatient_claims
    )
    WHERE next_admsn IS NOT NULL
  )
  GROUP BY CLM_DRG_CD
)
SELECT
  c.CLM_DRG_CD as drg,
  c.num_claims,
  c.total_cost,
  c.avg_cost_per_claim,
  c.avg_los,
  COALESCE(r.readmissions_30d, 0) as readmissions_30d,
  ROUND(COALESCE(r.readmissions_30d, 0) * 100.0 / COALESCE(r.total_admits, 1), 1) as readmission_rate_pct
FROM drg_costs c
LEFT JOIN drg_readmit r ON c.CLM_DRG_CD = r.CLM_DRG_CD
ORDER BY c.total_cost DESC
""", conn)

drg_summary.to_csv(EXPORT / "drg_summary.csv", index=False)
print(f"DRG summary: {len(drg_summary)} rows")
print(drg_summary.head(10))

---
## 2. Diagnosis-level summary (cost + volume)

In [ ]:
diagnosis_summary = pd.read_sql("""
SELECT
  ICD9_DGNS_CD_1 as diagnosis,
  COUNT(*) as num_claims,
  ROUND(SUM(CLM_PMT_AMT), 0) as total_cost,
  ROUND(AVG(CLM_PMT_AMT), 0) as avg_cost_per_claim,
  COUNT(DISTINCT DESYNPUF_ID) as unique_beneficiaries
FROM inpatient_claims
WHERE CLM_PMT_AMT > 0 AND ICD9_DGNS_CD_1 IS NOT NULL
GROUP BY ICD9_DGNS_CD_1
ORDER BY total_cost DESC
""", conn)

diagnosis_summary.to_csv(EXPORT / "diagnosis_summary.csv", index=False)
print(f"Diagnosis summary: {len(diagnosis_summary)} rows")
print(diagnosis_summary.head(10))

---
## 3. Beneficiary cohort segmentation (spending tier + condition count)

In [ ]:
bene = pd.read_sql("""
SELECT
  DESYNPUF_ID,
  YEAR,
  BENE_BIRTH_DT,
  BENE_DEATH_DT,
  BENE_SEX_IDENT_CD,
  MEDREIMB_IP + MEDREIMB_OP + MEDREIMB_CAR as total_spend,
  MEDREIMB_IP,
  MEDREIMB_OP,
  SP_DIABETES,
  SP_CHF,
  SP_COPD,
  SP_ISCHMCHT,
  SP_DEPRESSN,
  SP_ALZHDMTA,
  SP_CHRNKIDN,
  SP_OSTEOPRS,
  SP_RA_OA,
  SP_STRKETIA,
  SP_CNCR
FROM beneficiary_summary
""", conn)

chronic_cols = ['SP_DIABETES', 'SP_CHF', 'SP_COPD', 'SP_ISCHMCHT', 'SP_DEPRESSN',
                'SP_ALZHDMTA', 'SP_CHRNKIDN', 'SP_OSTEOPRS', 'SP_RA_OA', 'SP_STRKETIA', 'SP_CNCR']

bene['num_conditions'] = (bene[chronic_cols] == 1).sum(axis=1)

# Segment into spending tiers
bene['spending_tier'] = pd.cut(bene['total_spend'],
                                bins=[0, 500, 2000, 5000, 10000, 250000],
                                labels=['$0-500', '$500-2K', '$2K-5K', '$5K-10K', '$10K+'])

beneficiary_cohort = bene.copy()
beneficiary_cohort.to_csv(EXPORT / "beneficiary_cohort.csv", index=False)
print(f"Beneficiary cohort: {len(beneficiary_cohort)} rows")
print(f"\nSpending tier distribution:")
print(beneficiary_cohort['spending_tier'].value_counts().sort_index())

---
## 4. Chronic condition impact summary

In [ ]:
condition_impact = pd.DataFrame()

for condition in chronic_cols:
    with_cond = bene[bene[condition] == 1]
    without_cond = bene[bene[condition] == 2]
    
    row = {
        'condition': condition.replace('SP_', ''),
        'beneficiary_years_with_condition': len(with_cond),
        'prevalence_pct': round(len(with_cond) / len(bene) * 100, 1),
        'avg_spend_with': round(with_cond['total_spend'].mean()),
        'avg_spend_without': round(without_cond['total_spend'].mean()),
        'median_spend_with': round(with_cond['total_spend'].median()),
        'median_spend_without': round(without_cond['total_spend'].median())
    }
    row['spend_delta'] = row['avg_spend_with'] - row['avg_spend_without']
    condition_impact = pd.concat([condition_impact, pd.DataFrame([row])], ignore_index=True)

condition_impact = condition_impact.sort_values('spend_delta', ascending=False).reset_index(drop=True)
condition_impact.to_csv(EXPORT / "condition_impact.csv", index=False)
print("Chronic condition impact summary:")
print(condition_impact)

---
## 5. Comorbidity pairs (high-risk combinations)

In [ ]:
comorbidity_pairs = pd.DataFrame()

pairs = [
    ('SP_DIABETES', 'SP_CHF'),
    ('SP_DIABETES', 'SP_ISCHMCHT'),
    ('SP_CHF', 'SP_COPD'),
    ('SP_ISCHMCHT', 'SP_DEPRESSN'),
    ('SP_CHRNKIDN', 'SP_DIABETES'),
    ('SP_CHF', 'SP_ISCHMCHT'),
    ('SP_COPD', 'SP_DEPRESSN'),
]

for cond1, cond2 in pairs:
    both = bene[(bene[cond1] == 1) & (bene[cond2] == 1)]
    either = bene[((bene[cond1] == 1) | (bene[cond2] == 1))]
    neither = bene[(bene[cond1] == 2) & (bene[cond2] == 2)]
    
    row = {
        'condition_1': cond1.replace('SP_', ''),
        'condition_2': cond2.replace('SP_', ''),
        'count_both': len(both),
        'prevalence_both_pct': round(len(both) / len(bene) * 100, 1),
        'avg_spend_both': round(both['total_spend'].mean()),
        'avg_spend_either': round(either['total_spend'].mean()),
        'avg_spend_neither': round(neither['total_spend'].mean())
    }
    comorbidity_pairs = pd.concat([comorbidity_pairs, pd.DataFrame([row])], ignore_index=True)

comorbidity_pairs = comorbidity_pairs.sort_values('avg_spend_both', ascending=False).reset_index(drop=True)
comorbidity_pairs.to_csv(EXPORT / "comorbidity_pairs.csv", index=False)
print("High-risk comorbidity combinations:")
print(comorbidity_pairs)

---
## 6. Spending and readmission trends by year

In [ ]:
yearly_summary = pd.read_sql("""
SELECT
  YEAR,
  COUNT(DISTINCT DESYNPUF_ID) as num_beneficiaries,
  ROUND(AVG(MEDREIMB_IP), 0) as avg_inpatient_spend,
  ROUND(AVG(MEDREIMB_OP), 0) as avg_outpatient_spend,
  ROUND(AVG(MEDREIMB_IP + MEDREIMB_OP + MEDREIMB_CAR), 0) as avg_total_spend
FROM beneficiary_summary
GROUP BY YEAR
ORDER BY YEAR
""", conn)

yearly_summary.to_csv(EXPORT / "yearly_trends.csv", index=False)
print("Spending trends by year:")
print(yearly_summary)

---
## 7. Readmission summary by DRG

In [ ]:
readmission_by_drg = pd.read_sql("""
WITH readmit_flagged AS (
  SELECT
    DESYNPUF_ID,
    CLM_ADMSN_DT,
    NCH_BENE_DSCHRG_DT,
    CLM_DRG_CD,
    CLM_PMT_AMT,
    CAST((julianday(LEAD(CLM_ADMSN_DT) OVER (PARTITION BY DESYNPUF_ID ORDER BY CLM_ADMSN_DT)) - julianday(NCH_BENE_DSCHRG_DT)) AS INTEGER) as days_to_next_admit
  FROM inpatient_claims
  WHERE CLM_ADMSN_DT IS NOT NULL
)
SELECT
  CLM_DRG_CD as drg,
  COUNT(*) as num_admits,
  SUM(CASE WHEN days_to_next_admit > 0 AND days_to_next_admit <= 30 THEN 1 ELSE 0 END) as readmit_30d_count,
  ROUND(SUM(CASE WHEN days_to_next_admit > 0 AND days_to_next_admit <= 30 THEN 1 ELSE 0 END) * 100.0 / COUNT(*), 1) as readmit_30d_rate_pct,
  ROUND(AVG(CLM_PMT_AMT), 0) as avg_cost_per_admit
FROM readmit_flagged
GROUP BY CLM_DRG_CD
ORDER BY readmit_30d_rate_pct DESC
""", conn)

readmission_by_drg.to_csv(EXPORT / "readmission_by_drg.csv", index=False)
print(f"Readmission by DRG: {len(readmission_by_drg)} DRGs")
print(readmission_by_drg.head(15))

---
## 8. Summary statistics

In [ ]:
summary_stats = pd.DataFrame([
    {
        'metric': 'Total beneficiary-years',
        'value': len(bene)
    },
    {
        'metric': 'Unique beneficiaries',
        'value': bene['DESYNPUF_ID'].nunique()
    },
    {
        'metric': 'Total inpatient claims',
        'value': len(pd.read_sql("SELECT COUNT(*) as cnt FROM inpatient_claims", conn))
    },
    {
        'metric': 'Total outpatient claims',
        'value': len(pd.read_sql("SELECT COUNT(*) as cnt FROM outpatient_claims", conn))
    },
    {
        'metric': 'Total prescription events',
        'value': len(pd.read_sql("SELECT COUNT(*) as cnt FROM prescription_drug_events", conn))
    },
    {
        'metric': 'Total Medicare spend (billions)',
        'value': round(bene['total_spend'].sum() / 1e9, 2)
    },
    {
        'metric': 'Avg annual spend per beneficiary',
        'value': round(bene['total_spend'].mean())
    },
    {
        'metric': 'Avg inpatient readmission rate',
        'value': round(readmission_by_drg['readmit_30d_count'].sum() / readmission_by_drg['num_admits'].sum() * 100, 1)
    }
])

summary_stats.to_csv(EXPORT / "summary_statistics.csv", index=False)
print("\nProject Summary Statistics:")
print(summary_stats.to_string(index=False))

In [ ]:
conn.close()
print("\n" + "="*70)
print("Export complete. All files written to data/exports/")
print("="*70)
print("\nFiles ready for Tableau:")
import os
for f in sorted(os.listdir(EXPORT)):
    if f.endswith('.csv'):
        fsize = os.path.getsize(EXPORT / f) / 1024
        print(f"  {f:<40} {fsize:>8.1f} KB")